# 🎾 Padel ML — MLflow Experiment Tracking
### Tracking Classification + Regression models with MLflow

In [2]:
import subprocess, sys, os

# Désinstalle mlflow 3.11 conflictuel et réinstalle 2.9.2 proprement
env = os.environ.copy()
env['TEMP'] = 'D:\\tmp'
env['TMP'] = 'D:\\tmp'

# Utilise pip d'Anaconda directement
anaconda_pip = r'C:\Users\hammo\anaconda33\Scripts\pip.exe'

result = subprocess.run(
    [anaconda_pip, 'install', 'mlflow==2.9.2',
     '--no-cache-dir', '--cache-dir', 'D:/pip_cache',
     '--force-reinstall', '--no-deps'],
    capture_output=True, text=True, env=env
)
print(result.stdout[-500:] if result.stdout else '')
print(result.stderr[-300:] if result.stderr else '')
print('Done!')

  Using cached mlflow-2.9.2-py3-none-any.whl (19.1 MB)
  Attempting uninstall: mlflow
    Found existing installation: mlflow 2.9.2
    Uninstalling mlflow-2.9.2:
      Successfully uninstalled mlflow-2.9.2


Done!


In [3]:
import sys

# Force uniquement Anaconda
sys.path = [p for p in sys.path 
            if 'Python312' not in p 
            and 'python312_packages' not in p]

# Ajoute Anaconda en premier
sys.path.insert(0, r'C:\Users\hammo\anaconda33\Lib\site-packages')

import mlflow
print('MLflow version:', mlflow.__version__)
print('✅ Ready!')

MLflow version: 2.9.2
✅ Ready!


In [4]:
# CELL 3 — Load and prepare data
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBClassifier
import joblib

df = pd.read_csv('../data/clean_matches.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

df2 = df.dropna(subset=['winner']).copy()
df2['target'] = (df2['winner'] == 'team_1').astype(int)

le_cat   = LabelEncoder().fit(df2['category'])
le_rn    = LabelEncoder().fit(df2['round_name'])
le_court = LabelEncoder().fit(df2['court'].fillna('Unknown'))
le_src   = LabelEncoder().fit(df2['competition_source'])

df2['category_enc']   = le_cat.transform(df2['category'])
df2['round_name_enc'] = le_rn.transform(df2['round_name'])
df2['court_enc']      = le_court.transform(df2['court'].fillna('Unknown'))
df2['source_enc']     = le_src.transform(df2['competition_source'])
df2['played_at']      = pd.to_datetime(df2['played_at'], errors='coerce')
df2['month']          = df2['played_at'].dt.month.fillna(0).astype(int)

FEATURES = ['category_enc','round','round_name_enc','index','court_enc','source_enc','month']
X = df2[FEATURES]
y = df2['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print('✅ Data ready!')

Shape: (1260, 16)
Columns: ['id', 'name', 'category', 'round', 'round_name', 'index', 'played_at', 'status', 'score', 'winner', 'duration_minutes', 'players', 'competition_source', 'season_year', 'tournament_name', 'court']
Train: (915, 7) | Test: (229, 7)
✅ Data ready!


In [5]:
# CELL 4 — MLflow Run 1: Random Forest Classifier
with mlflow.start_run(run_name='RandomForest_Classification_v1'):
    # Parameters
    params = {
        'n_estimators': 100,
        'max_depth': None,
        'random_state': 42,
        'model_type': 'RandomForestClassifier',
        'task': 'classification',
        'dataset': 'clean_matches.csv',
        'features': str(FEATURES)
    }
    mlflow.log_params(params)
    
    # Train
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    # Metrics
    preds = rf.predict(X_test)
    proba = rf.predict_proba(X_test)[:,1]
    acc   = accuracy_score(y_test, preds)
    auc   = roc_auc_score(y_test, proba)
    
    mlflow.log_metric('accuracy', round(acc, 4))
    mlflow.log_metric('roc_auc',  round(auc, 4))
    mlflow.log_metric('train_size', len(X_train))
    mlflow.log_metric('test_size',  len(X_test))
    
    # Save model
    mlflow.sklearn.log_model(rf, 'random_forest_classifier')
    
    # Tags
    mlflow.set_tag('model', 'RandomForest')
    mlflow.set_tag('task', 'winner_prediction')
    mlflow.set_tag('author', 'Yassine Hammouda')
    
    run_id_rf = mlflow.active_run().info.run_id
    print(f'✅ Run 1 — Random Forest')
    print(f'   Accuracy : {acc:.4f} ({acc*100:.1f}%)')
    print(f'   ROC-AUC  : {auc:.4f}')
    print(f'   Run ID   : {run_id_rf}')

2026/04/27 02:14:07 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



✅ Run 1 — Random Forest
   Accuracy : 0.5371 (53.7%)
   ROC-AUC  : 0.6025
   Run ID   : 8cfe78e5e9ca4b3fb1fac284da7df085


C:\Users\hammo\anaconda33\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [6]:
# CELL 5 — MLflow Run 2: XGBoost Classifier
with mlflow.start_run(run_name='XGBoost_Classification_v1'):
    # Parameters
    params = {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.3,
        'random_state': 42,
        'model_type': 'XGBClassifier',
        'task': 'classification',
        'dataset': 'clean_matches.csv',
        'features': str(FEATURES)
    }
    mlflow.log_params(params)
    
    # Train
    xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
    xgb.fit(X_train, y_train)
    
    # Metrics
    preds = xgb.predict(X_test)
    proba = xgb.predict_proba(X_test)[:,1]
    acc   = accuracy_score(y_test, preds)
    auc   = roc_auc_score(y_test, proba)
    
    mlflow.log_metric('accuracy', round(acc, 4))
    mlflow.log_metric('roc_auc',  round(auc, 4))
    mlflow.log_metric('train_size', len(X_train))
    mlflow.log_metric('test_size',  len(X_test))
    
    # Save model
    mlflow.xgboost.log_model(xgb, 'xgboost_classifier')
    
    # Tags
    mlflow.set_tag('model', 'XGBoost')
    mlflow.set_tag('task', 'winner_prediction')
    mlflow.set_tag('winner', 'True')
    mlflow.set_tag('author', 'Yassine Hammouda')
    
    run_id_xgb = mlflow.active_run().info.run_id
    print(f'✅ Run 2 — XGBoost')
    print(f'   Accuracy : {acc:.4f} ({acc*100:.1f}%)')
    print(f'   ROC-AUC  : {auc:.4f}')
    print(f'   Run ID   : {run_id_xgb}')

C:\Users\hammo\anaconda33\lib\site-packages\xgboost\core.py:158: UserWarning: [02:14:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


✅ Run 2 — XGBoost
   Accuracy : 0.7424 (74.2%)
   ROC-AUC  : 0.7935
   Run ID   : 715d6d086a37461a8eca70484f835b7b


In [7]:
# CELL 6 — MLflow Run 3: Random Forest Regressor (Duration)
df3 = df.dropna(subset=['duration_minutes', 'winner']).copy()
le_cat2   = LabelEncoder().fit(df3['category'])
le_rn2    = LabelEncoder().fit(df3['round_name'])
le_court2 = LabelEncoder().fit(df3['court'].fillna('Unknown'))
le_src2   = LabelEncoder().fit(df3['competition_source'])
df3['category_enc']   = le_cat2.transform(df3['category'])
df3['round_name_enc'] = le_rn2.transform(df3['round_name'])
df3['court_enc']      = le_court2.transform(df3['court'].fillna('Unknown'))
df3['source_enc']     = le_src2.transform(df3['competition_source'])
df3['winner_enc']     = (df3['winner'] == 'team_1').astype(int)
df3['played_at']      = pd.to_datetime(df3['played_at'], errors='coerce')
df3['month']          = df3['played_at'].dt.month.fillna(0).astype(int)

FEATURES_REG = ['category_enc','round','round_name_enc','index','court_enc','source_enc','month','winner_enc']
X_reg = df3[FEATURES_REG]
y_reg = df3['duration_minutes']
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

with mlflow.start_run(run_name='RandomForest_Regression_Duration_v1'):
    params = {
        'n_estimators': 100,
        'random_state': 42,
        'model_type': 'RandomForestRegressor',
        'task': 'regression',
        'target': 'duration_minutes',
        'dataset': 'clean_matches.csv'
    }
    mlflow.log_params(params)
    
    rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_reg.fit(X_tr, y_tr)
    preds_reg = rf_reg.predict(X_te)
    
    mae  = mean_absolute_error(y_te, preds_reg)
    rmse = mean_squared_error(y_te, preds_reg)**0.5
    r2   = r2_score(y_te, preds_reg)
    
    mlflow.log_metric('mae',  round(mae, 3))
    mlflow.log_metric('rmse', round(rmse, 3))
    mlflow.log_metric('r2',   round(r2, 4))
    mlflow.log_metric('train_size', len(X_tr))
    
    mlflow.sklearn.log_model(rf_reg, 'rf_regressor_duration')
    mlflow.set_tag('model', 'RandomForestRegressor')
    mlflow.set_tag('task', 'duration_prediction')
    mlflow.set_tag('author', 'Yassine Hammouda')
    
    print(f'✅ Run 3 — RF Regressor (Duration)')
    print(f'   MAE  : {mae:.2f} min')
    print(f'   RMSE : {rmse:.2f} min')
    print(f'   R²   : {r2:.4f}')

✅ Run 3 — RF Regressor (Duration)
   MAE  : 26.27 min
   RMSE : 32.05 min
   R²   : -0.1814


In [8]:
import os

# Les runs sont dans mlruns/ du dossier notebooks/
local_uri = 'file:///' + os.path.abspath('mlruns').replace('\\', '/')
print('Local runs URI:', local_uri)

client = mlflow.tracking.MlflowClient(tracking_uri=local_uri)
all_experiments = client.search_experiments()
print(f'Experiments: {[e.name for e in all_experiments]}')

all_runs = []
for exp in all_experiments:
    runs = client.search_runs(exp.experiment_id)
    all_runs.extend(runs)

print(f'Total runs: {len(all_runs)}')
print('=' * 50)

for run in all_runs:
    name = run.data.tags.get('mlflow.runName', 'Unknown')
    metrics = run.data.metrics
    print(f'\n✅ {name}')
    if 'accuracy' in metrics:
        print(f'   Accuracy : {metrics["accuracy"]*100:.1f}%')
        print(f'   ROC-AUC  : {metrics["roc_auc"]:.4f}')
    if 'mae' in metrics:
        print(f'   MAE  : {metrics["mae"]:.2f} min')
        print(f'   R²   : {metrics["r2"]:.4f}')

print('\n✅ MLflow tracking complete!')

Local runs URI: file:///C:/Users/hammo/Downloads/BI_ML_Project/notebooks/mlruns
Experiments: ['Default']
Total runs: 6

✅ RandomForest_Regression_Duration_v1
   MAE  : 26.27 min
   R²   : -0.1814

✅ XGBoost_Classification_v1
   Accuracy : 74.2%
   ROC-AUC  : 0.7935

✅ RandomForest_Classification_v1
   Accuracy : 53.7%
   ROC-AUC  : 0.6025

✅ RandomForest_Regression_Duration_v1
   MAE  : 26.27 min
   R²   : -0.1814

✅ XGBoost_Classification_v1
   Accuracy : 74.2%
   ROC-AUC  : 0.7935

✅ RandomForest_Classification_v1
   Accuracy : 53.7%
   ROC-AUC  : 0.6025

✅ MLflow tracking complete!


In [9]:
# CELL 8 — Best model summary
print('=' * 50)
print('🏆 BEST MODEL SUMMARY')
print('=' * 50)
print('Task: Match Winner Prediction')
print('Winner: XGBoost Classifier')
print('  Accuracy : 74.2%')
print('  ROC-AUC  : 0.7935')
print()
print('📁 MLflow runs stored in: D:/mlflow_runs')
print('🚀 View UI: mlflow ui --backend-store-uri D:/mlflow_runs --port 5001')
print()
print('✅ MLflow tracking complete!')
print('✅ 3 runs tracked and comparable')
print('✅ Models saved as artifacts')

🏆 BEST MODEL SUMMARY
Task: Match Winner Prediction
Winner: XGBoost Classifier
  Accuracy : 74.2%
  ROC-AUC  : 0.7935

📁 MLflow runs stored in: D:/mlflow_runs
🚀 View UI: mlflow ui --backend-store-uri D:/mlflow_runs --port 5001

✅ MLflow tracking complete!
✅ 3 runs tracked and comparable
✅ Models saved as artifacts


In [10]:
# Run 4 — XGBoost Regressor
from xgboost import XGBRegressor

with mlflow.start_run(run_name='XGBoost_Regression_Duration_v2'):
    params = {
        'n_estimators': 100,
        'max_depth': 4,
        'learning_rate': 0.1,
        'model_type': 'XGBRegressor',
        'task': 'regression',
        'target': 'duration_minutes'
    }
    mlflow.log_params(params)
    
    xgb_reg = XGBRegressor(n_estimators=100, max_depth=4, random_state=42, verbosity=0)
    xgb_reg.fit(X_tr, y_tr)
    preds = xgb_reg.predict(X_te)
    
    mae  = mean_absolute_error(y_te, preds)
    rmse = mean_squared_error(y_te, preds)**0.5
    r2   = r2_score(y_te, preds)
    
    mlflow.log_metric('mae',  round(mae, 3))
    mlflow.log_metric('rmse', round(rmse, 3))
    mlflow.log_metric('r2',   round(r2, 4))
    mlflow.set_tag('model', 'XGBRegressor')
    mlflow.set_tag('author', 'Yassine Hammouda')
    
    print(f'✅ Run 4 — XGBoost Regressor')
    print(f'   MAE  : {mae:.2f} min')
    print(f'   RMSE : {rmse:.2f} min')
    print(f'   R²   : {r2:.4f}')

✅ Run 4 — XGBoost Regressor
   MAE  : 28.13 min
   RMSE : 34.10 min
   R²   : -0.3377
